# Publish the Qwen3 GGUF files as Kaggle Models

Copies selected files from my Hugging Face repos into Kaggle Models with `kagglehub`. Kaggle's network does
the transfer, so nothing is uploaded from a home connection.

**One-time setup:** Add-ons → Secrets → add `KAGGLE_USERNAME` and `KAGGLE_KEY` (from kaggle.com → Settings →
API → Create New Token). Internet must be on. Run one model at a time (the disk is limited).

After the upload, open each model page on Kaggle and paste the description from the drafts.

In [ ]:
!pip install -q -U kagglehub huggingface_hub
import os, shutil
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")
import kagglehub
from huggingface_hub import hf_hub_download

In [ ]:
OWNER = os.environ["KAGGLE_USERNAME"]
PLAN = {  # Kaggle model slug: (HF repo, files per variation)
    "qwen3-8b-gguf": ("muhammed-afsal-p-m/Qwen3-8B-GGUF",
                      {"q8_0": ["Qwen3-8B-Q8_0.gguf"], "q4_k_m": ["Qwen3-8B-Q4_K_M.gguf"],
                       "iq4_xs": ["Qwen3-8B-IQ4_XS.gguf"]}),
    "qwen3-14b-gguf": ("muhammed-afsal-p-m/Qwen3-14B-GGUF",
                       {"q8_0": ["Qwen3-14B-Q8_0.gguf"], "q4_k_m": ["Qwen3-14B-Q4_K_M.gguf"],
                        "iq3_m": ["Qwen3-14B-IQ3_M.gguf"]}),
    "qwen3.8-27b-gguf": ("muhammed-afsal-p-m/Qwen3.8-27B-GGUF",
                         {"q4_k_m": ["Qwen3.8-27B-Q4_K_M.gguf", "mmproj-Qwen3.8-27B-F16.gguf"],
                          "iq3_m": ["Qwen3.8-27B-IQ3_M.gguf", "mmproj-Qwen3.8-27B-F16.gguf"]}),
}
MODEL_SLUG = "qwen3-8b-gguf"   # run once per model: qwen3-8b-gguf, qwen3-14b-gguf, qwen3.8-27b-gguf

In [ ]:
repo, variations = PLAN[MODEL_SLUG]
for variation, files in variations.items():
    folder = f"/kaggle/tmp/{MODEL_SLUG}/{variation}"
    os.makedirs(folder, exist_ok=True)
    for f in files + ["SHA256SUMS.txt"]:
        hf_hub_download(repo, f, local_dir=folder)
    kagglehub.model_upload(f"{OWNER}/{MODEL_SLUG}/gguf/{variation}", folder, license_name="Apache 2.0",
                           version_notes=f"From https://huggingface.co/{repo}")
    shutil.rmtree(folder)  # free the disk before the next variation
    print("uploaded", variation)
print(f"https://www.kaggle.com/models/{OWNER}/{MODEL_SLUG}")

If Kaggle rejects the `gguf` framework name, change `/gguf/` to `/other/` in the handle above and run again.